# Preprocessing

Turns the findings from `01_eda.ipynb` into a model-ready dataset.

In [28]:
import numpy as np
import pandas as pd

# readable numbers: no scientific notation; 1 decimal for large values, 3 for [0,1] scores
pd.set_option("display.float_format", lambda x: f"{x:,.1f}" if abs(x) >= 1 else f"{x:.3f}")

RANDOM_STATE = 42
DATA_DIR = "../data/raw"

df = pd.read_csv(f"{DATA_DIR}/application_train.csv")
df.shape

(307511, 122)

## 1. Pre-split cleaning

Fixed rules only: known bad values removed, a documented placeholder recoded, columns dropped on criteria the EDA already settled.

### 1a. Dropping columns

- **Redundant copies** — each building measurement appears three times as `_AVG`/`_MEDI`/`_MODE`; only `_AVG` is kept. `TOTALAREA_MODE` and `EMERGENCYSTATE_MODE` are spared — neither has any duplicates.
- **`SK_ID_CURR`** — an application reference number, with no predictive content.
- **Highly correlated features**:
  - `DEF_60_CNT_SOCIAL_CIRCLE` (r = 0.861 with `DEF_30`). The 30-day definition records a non-zero count for 11.4% of clients against 8.4%; `DEF_60` is zero for 92% of the file, leaving less of it able to carry signal.
  - `OBS_60_CNT_SOCIAL_CIRCLE` (r = 0.998 with `OBS_30`) — indistinguishable from its twin on every measure, dropping either is safe.
  - `REGION_RATING_CLIENT` (r = 0.951) — the same rating as `REGION_RATING_CLIENT_W_CITY` without the city taken into account, so the finer of the two is kept.
  - `CNT_CHILDREN`/`CNT_FAM_MEMBERS` correlate 0.879 but both stay: their difference is exactly one or two adults, which no other column records.
  - `AMT_CREDIT`/`AMT_GOODS_PRICE` correlate at 0.99 — the loan is almost always the price of the financed goods.

In [29]:
features = df.drop(columns="TARGET")

# _MEDI or _MODE columns are redundant only when the same measurement also exists as _AVG
avg_measurements = {c[:-4] for c in features.columns if c.endswith("_AVG")}
redundant = {c for c in features.columns 
             if c.endswith(("_MEDI", "_MODE")) and c[:-5] in avg_measurements
}

# Highly correlated features
near_duplicates = {"DEF_60_CNT_SOCIAL_CIRCLE", "OBS_60_CNT_SOCIAL_CIRCLE", "REGION_RATING_CLIENT", "AMT_GOODS_PRICE"}

drop_cols = sorted(redundant | near_duplicates | {"SK_ID_CURR"})
df = df.drop(columns=drop_cols)


print(f"redundant copies      : {len(redundant):>3}")
print(f"identifier            : {1:>3}")
print(f"near-duplicates       : {len(near_duplicates):>3}")
print(f"dropped in total      : {len(drop_cols):>3}")
print(f"columns remaining     : {df.shape[1]:>3}   (including TARGET)")

redundant copies      :  28
identifier            :   1
near-duplicates       :   4
dropped in total      :  33
columns remaining     :  89   (including TARGET)


### 1b. Dropping rows

`CODE_GENDER = 'XNA'` (4 rows) and `NAME_FAMILY_STATUS = 'Unknown'` (2 rows) are placeholder categories; `AMT_INCOME_TOTAL = 117,000,000` (1 row) is implausible against a median of 147,150.

In [30]:
n_before = len(df)

df = df[
    (df["CODE_GENDER"] != "XNA")
    & (df["NAME_FAMILY_STATUS"] != "Unknown")
    & (df["AMT_INCOME_TOTAL"] != 117_000_000)
].reset_index(drop=True)

print(f"dropped {n_before - len(df)} rows ({n_before:,} -> {len(df):,})")

dropped 7 rows (307,511 -> 307,504)


### 1c. The `DAYS_EMPLOYED` placeholder

The column is documented as *days before the application that the client started current employment*, so every genuine value is negative. About 18% of rows hold `365243` instead — positive, roughly a thousand years — and those rows are pensioners plus every unemployed client in the data.

The definition presupposes current employment, so replacing it with `0` would claim the client started work on the application date. By contrast, `NaN` is the only value meaning "not applicable".

This value must be replaced before performing imputation. Otherwise, the large number of occurrences of this code will distort the median.

In [31]:
placeholder = df["DAYS_EMPLOYED"] == 365243

print(f"placeholder rows: {placeholder.sum():,} ({placeholder.mean():.1%})")
print(df.loc[placeholder, "NAME_INCOME_TYPE"].value_counts().to_string())

df['DAYS_EMPLOYED'] = df['DAYS_EMPLOYED'].replace(365243, np.nan)

print(f"\nDAYS_EMPLOYED is now {df['DAYS_EMPLOYED'].isna().mean():.1%} missing, to be filled at imputation")

placeholder rows: 55,374 (18.0%)
NAME_INCOME_TYPE
Pensioner     55352
Unemployed       22

DAYS_EMPLOYED is now 18.0% missing, to be filled at imputation


## 2. Feature engineering

Everything below is arithmetic within a single row, involving no statistic pooled across clients, so it still precedes the split.

### 2a. Categorical cleaning and grouping

In [32]:
# ===================================================================
# ========================= NAME_TYPE_SUITE =========================
# ===================================================================

df["NAME_TYPE_SUITE"] = df["NAME_TYPE_SUITE"].fillna("Missing")
df["NAME_TYPE_SUITE"] = df["NAME_TYPE_SUITE"].replace(
    {
        "Other_A": "Other",
        "Other_B": "Other",
        "Group of people": "Other"
    }
)

# ===================================================================
# ========================= NAME_INCOME_TYPE ========================
# ===================================================================

rare_income = ["Unemployed", "Student", "Businessman", "Maternity leave"]

df["NAME_INCOME_TYPE"] = df["NAME_INCOME_TYPE"].replace(
    rare_income,
    "Other"
)

# ===================================================================
# ===================== OCCUPATION_TYPE_GROUPED =====================
# ===================================================================

def group_occupation(x):

    # Missing / non-observed occupation
    if pd.isna(x):
        return "Missing"

    # 1. Labor / Low-skill
    elif x in ["Laborers", "Low-skill Laborers"]:
        return "Labor / Low-skill"

    # 2. Sales / Service
    elif x in ["Sales staff", "Private service staff", "Waiters/barmen staff", 
               "Cooking staff", "Cleaning staff"]:
        return "Sales / Service"

    # 3. Core / Administrative
    elif x in ["Core staff", "Accountants", "Secretaries", "HR staff", 
               "Medicine staff", "Realty agents", "Security staff", "Drivers"]:
        return "Core / Administrative"

    # 4. High-skill / Managerial
    elif x in ["High skill tech staff", "Managers", "IT staff"]:
        return "High-skill / Managerial"

    # Fallback
    else:
        return "Other"

df["OCCUPATION_TYPE"] = df["OCCUPATION_TYPE"].apply(group_occupation)

# ===================================================================
# ==================== ORGANIZATION_TYPE_GROUPED ====================
# ===================================================================

def group_organization(x):

    # no NaN in this file, but a future applicant could arrive with one,
    # and .startswith would raise on it
    if pd.isna(x):
        return "Missing"

    elif x.startswith("Business Entity"):
        return "Business"

    elif x == "Self-employed":
        return "Self-employed"

    elif x.startswith("Industry:"):
        return "Industry"

    elif x.startswith("Trade:"):
        return "Trade"

    elif x.startswith("Transport:"):
        return "Transport"

    elif x in ["School", "Kindergarten", "University"]:
        return "Education"

    elif x == "Medicine":
        return "Healthcare"

    elif x == "Government":
        return "Government & Public"

    elif x in ["Security", "Security Ministries", "Military", "Police"]:
        return "Security & Military"

    elif x in ["Construction", "Housing", "Realtor"]:
        return "Construction & Housing"

    elif x in ["Hotel", "Restaurant"]:
        return "Hospitality & Food"

    elif x in ["Bank", "Insurance", "Legal Services","Advertising", "Services"]:
        return "Financial & Professional Services"

    elif x in ["Electricity", "Telecom", "Mobile", "Postal"]:
        return "Utilities & Communication"

    elif x == "XNA":
        return "Missing"

    else:
        return "Other"

df["ORGANIZATION_TYPE"] = df["ORGANIZATION_TYPE"].apply(group_organization)

# ===================================================================
# ======================= EMERGENCYSTATE_MODE =======================
# ===================================================================

df["EMERGENCYSTATE_MODE"] = df["EMERGENCYSTATE_MODE"].fillna("Missing")

### 2b. Converting the `DAYS_` columns to years

In [33]:
DAYS_TO_YEARS = {
    "DAYS_BIRTH": "AGE_YEARS",
    "DAYS_EMPLOYED": "EMPLOYMENT_YEARS",
    "DAYS_REGISTRATION": "YEARS_SINCE_REGISTRATION",
    "DAYS_ID_PUBLISH": "YEARS_SINCE_ID_PUBLISH",
    "DAYS_LAST_PHONE_CHANGE": "YEARS_SINCE_PHONE_CHANGE",
}

# every value must already be a count backwards, or negating would silently flip it
assert (df[list(DAYS_TO_YEARS)].max() <= 0).all(), "a DAYS_ column holds a positive value"

years = pd.DataFrame({new: df[old].abs() / 365.25 for old, new in DAYS_TO_YEARS.items()})
df = pd.concat([df.drop(columns=list(DAYS_TO_YEARS)), years], axis=1)

print(df[list(DAYS_TO_YEARS.values())].describe().round(2).to_string())

       AGE_YEARS  EMPLOYMENT_YEARS  YEARS_SINCE_REGISTRATION  YEARS_SINCE_ID_PUBLISH  YEARS_SINCE_PHONE_CHANGE
count  307,504.0         252,130.0                 307,504.0               307,504.0                 307,503.0
mean        43.9               6.5                      13.7                     8.2                       2.6
std         11.9               6.4                       9.7                     4.1                       2.3
min         20.5             0.000                     0.000                   0.000                     0.000
25%         34.0               2.1                       5.5                     4.7                     0.750
50%         43.1               4.5                      12.3                     8.9                       2.1
75%         53.9               8.7                      20.5                    11.8                       4.3
max         69.1              49.0                      67.5                    19.7                      11.8


### 2c. Ratio features

Lending is underwritten on capacity to repay: the same 500,000 loan is routine for one
applicant and unpayable for another. `AMT_CREDIT` alone cannot tell them apart. Two standard
affordability measures can — debt-to-income and payment-to-income:

- `CREDIT_INCOME_RATIO` — the loan in years of gross income.
- `ANNUITY_INCOME_RATIO` — the share of annual income each instalment consumes.

The source columns stay: a ratio discards the levels, so 100k on 50k of income and 400k on
200k both score 2.0.

In [34]:
ratios = pd.DataFrame({
    "CREDIT_INCOME_RATIO": df["AMT_CREDIT"] / df["AMT_INCOME_TOTAL"],
    "ANNUITY_INCOME_RATIO": df["AMT_ANNUITY"] / df["AMT_INCOME_TOTAL"],
})
df = pd.concat([df, ratios], axis=1)

print(df[["CREDIT_INCOME_RATIO", "ANNUITY_INCOME_RATIO"]].describe().round(2).to_string())
print(f"\nfinal shape: {df.shape}")

       CREDIT_INCOME_RATIO  ANNUITY_INCOME_RATIO
count            307,504.0             307,492.0
mean                   4.0                 0.180
std                    2.7                 0.090
min                  0.040                 0.000
25%                    2.0                 0.110
50%                    3.3                 0.160
75%                    5.2                 0.230
max                   84.7                   1.9

final shape: (307504, 91)


### 2d. Missingness indicator

Imputation fills every gap, erasing the fact that a value was absent. Where that absence is informative, an `IS_MISSING_` flag records it first. Numeric gaps are then filled with the median, categorical ones with a `"Missing"` category.

`TOTALAREA_MODE`, `FLOORSMAX_AVG` and `YEARS_BEGINEXPLUATATION_AVG` are mostly missing together, so only one indicator is created for all of them.

The categorical columns need nothing. `SimpleImputer(strategy="constant", fill_value="Missing")` gives them an explicit `Missing` level, and once one-hot encoded that level *is* a missingness indicator.

In [35]:
cols = ['TOTALAREA_MODE', 'FLOORSMAX_AVG', 'YEARS_BEGINEXPLUATATION_AVG']
check_miss = df[cols].isna().all(axis=1).sum() / df[cols].isna().any(axis=1).sum() * 100
print(f"{check_miss:.1f}% of the rows that have missing data in at least one of these three columns have missing data in all three columns.")

96.2% of the rows that have missing data in at least one of these three columns have missing data in all three columns.


In [36]:
INDICATE_MISSING = ["EXT_SOURCE_1", "EXT_SOURCE_3", "TOTALAREA_MODE", "AMT_REQ_CREDIT_BUREAU_MON"]

indicators = pd.DataFrame({
    f"IS_MISSING_{col}": df[col].isna().astype(int) for col in INDICATE_MISSING
})
df = pd.concat([df, indicators], axis=1)
df = df.rename(columns={"IS_MISSING_AMT_REQ_CREDIT_BUREAU_MON": "IS_MISSING_AMT_REQ_CREDIT_BUREAU"})

print(f"\nshape: {df.shape}")


shape: (307504, 95)


## 3. Train / test split

The data is cut 80/20, stratified on the target so both sets carry the same 8.07% default rate.

In [37]:
from sklearn.model_selection import train_test_split

TEST_SIZE = 0.20

X = df.drop(columns="TARGET")
y = df["TARGET"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE
)

for name, y_part in [("train", y_train), ("test", y_test)]:
    print(f"{name:<6}{len(y_part):>8,} rows   {len(y_part)/len(y):>5.1%} of total   "
          f"default rate {y_part.mean():.4f}")

train  246,003 rows   80.0% of total   default rate 0.0807
test    61,501 rows   20.0% of total   default rate 0.0807


### 3.1. Post-split cleaning

- **Above 50% missing** — too little left to impute from. One exception: `EXT_SOURCE_1`.
- **Rare binaries** — a 0/1 column where the minority class represents less than 1% of total clients.
- **Constant** — one value in the training fold, so nothing to learn from.

All three rules are computed on `X_train` alone and the resulting column list is applied to both folds, so the test set never influences which features survive.

In [38]:
# Missing
MISSING_THRESHOLD = 50
missing_pct = X_train.isna().mean() * 100
too_empty = set(missing_pct[missing_pct > MISSING_THRESHOLD].index) - {"EXT_SOURCE_1"}

# 0/1 unbalanced
rare_flags = {c for c in X_train.columns if X_train[c].nunique() == 2 and X_train[c].value_counts(normalize=True).min() < 0.01}

# Constant features
constant = {c for c in X_train.columns if X_train[c].nunique() == 1}

print(f"more than {MISSING_THRESHOLD}% empty : {len(too_empty):>3}")
print(f"rare binaries : {len(rare_flags):>3}")
print(f"constant features : {len(constant):>3}")

# Drop
drop_cols = sorted(too_empty | rare_flags | constant)
X_train = X_train.drop(columns=drop_cols)
X_test = X_test.drop(columns=drop_cols)

# Check
print("=========================")
print(f"Train: {X_train.shape}")
print(f"Test: {X_test.shape}")

more than 50% empty :  16
rare binaries :  18
constant features :   0
Train: (246003, 60)
Test: (61501, 60)


## 4. Building the preprocessor

Two preprocessors are built, because the models fall into two families with different requirements.

**Logistic Regression** requires missing values to be handled explicitly and numerical features on a comparable scale. Lastly, categorical variables need to be converted into numerical representations (0/1).

**The four tree-based models** generally do not require feature scaling, because their decisions are based on threshold comparisons rather than distances or linear combinations. More importantly, XGBoost and LightGBM treat a missing value as a *third direction* at each split, sending those rows down whichever branch reduces the loss — filling the gap with a median destroys information they are built to use. Both can also split on a subset of categories directly, which one-hot encoding prevents; LightGBM's own documentation advises against feeding it dummy columns.

| | `preprocessor_linear` | `preprocessor_tree` |
|---|---|---|
| models | LogReg, LogReg Elastic Net | Decision Tree, Random Forest, XGBoost, LightGBM |
| numeric | median impute → scale | passthrough, gaps kept |
| ordinal | encode → scale | encode |
| categorical | one-hot, `drop="first"` | integer codes |
| flags | passthrough | passthrough |

### 4a. Column groups

`FLAG_OWN_CAR` and `FLAG_OWN_REALTY` are named like the binary `FLAG_DOCUMENT_*` columns, but hold `Y`/`N` text; selecting on the prefix would send text into a numeric branch. Grouping is therefore by dtype, not by name. Nothing is lost: one-hot encoding a two-valued column with `drop="first"` returns the same 1/0.

**`NAME_EDUCATION_TYPE` is separated out** because its categories have a real order that one-hot encoding would discard.

Both preprocessors use the four groups defined below.

In [39]:
ORDINAL_COLS = ["NAME_EDUCATION_TYPE"]

text_cols = X_train.select_dtypes(exclude="number").columns.tolist()

categorical_cols = [c for c in text_cols if c not in ORDINAL_COLS]
flag_cols = [c for c in X_train.columns if c.startswith("FLAG_") and c not in text_cols]
numeric_cols = [c for c in X_train.columns if c not in text_cols + flag_cols]

print(f"{'ordinal':<14}{len(ORDINAL_COLS):>4}")
print(f"{'categorical':<14}{len(categorical_cols):>4}")
print(f"{'binary flags':<14}{len(flag_cols):>4}")
print(f"{'numeric':<14}{len(numeric_cols):>4}")

ordinal          1
categorical     12
binary flags     8
numeric         39


In [40]:
EDUCATION_ORDER = [
    "Lower secondary",
    "Secondary / secondary special",
    "Incomplete higher",
    "Higher education",
    "Academic degree",
]

by_education = y_train.groupby(X_train["NAME_EDUCATION_TYPE"]).agg(["mean", "size"])
by_education = by_education.reindex(EDUCATION_ORDER)
by_education["mean"] = (by_education["mean"] * 100).round(2)

print(by_education.rename(columns={"mean": "default_%", "size": "n"}).to_string())

                               default_%       n
NAME_EDUCATION_TYPE                             
Lower secondary                     11.1    3060
Secondary / secondary special        9.0  174686
Incomplete higher                    8.2    8249
Higher education                     5.3   59881
Academic degree                      2.4     127


### 4b. The numeric branch (linear only)

Two learned operations: a median to fill gaps, a mean and standard deviation to scale.

**Scaling** matters for the linear models: a regularisation penalty treats all coefficients alike, so an unscaled feature measured in hundreds of thousands would be penalised out for reasons unrelated to its value. The tree models are unaffected either way.

**Imputation is the median, uniformly.** `EMPLOYMENT_YEARS` is 18% missing and those gaps are pensioners, who therefore receive about four and a half years of employment — the tenure of a group that defaults at 8.65%, when pensioners themselves default at 5.43%. For the linear models the fact is not lost, because `NAME_INCOME_TYPE` and `ORGANIZATION_TYPE` both still record it. The tree preprocessor avoids the substitution altogether by keeping the gap.

In [41]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

numeric_branch = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
])

### 4c. The text branches (linear only)

**Ordinal — `NAME_EDUCATION_TYPE`** → 0–4 in the verified order, from an explicit list rather than alphabetically. No missing values, so no imputer. Scaled with everything else.

**Nominal — the other 12** → one 0/1 column per category, with four settings:
- `SimpleImputer(constant, "Missing")` — absence as a category. `OCCUPATION_TYPE` is 31% blank, largely pensioners; a mode would invent a job.
- `min_frequency=800` — merges categories holding fewer than 800 clients. It is what stops a rare level in *future* data from getting its own column.
- `handle_unknown="infrequent_if_exist"` — an unseen category at prediction time joins that bucket instead of raising.
- `drop="first"` — drops one level per column as the reference, so the dummies are not collinear with the intercept.

In [42]:
from sklearn.base import clone
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder

ordinal_branch = Pipeline([
    ("encode", OrdinalEncoder(
        categories=[EDUCATION_ORDER],
        handle_unknown="use_encoded_value",
        unknown_value=-1,
    )),
    ("scale", StandardScaler()),
])

categorical_branch = Pipeline([
    ("impute", SimpleImputer(strategy="constant", fill_value="Missing")),
    ("encode", OneHotEncoder(
        min_frequency=800,
        handle_unknown="infrequent_if_exist",
        drop="first",
        sparse_output=False,
    )),
])

# where min_frequency actually merged categories (on a copy, so the branch above stays unfitted)
encoder = clone(categorical_branch).fit(X_train[categorical_cols]).named_steps["encode"]

merged = pd.DataFrame(
    [
        {"categories": len(cats), "merged": len(infrequent), "columns out": len(cats) - len(infrequent)}
        for cats, infrequent in zip(encoder.categories_, encoder.infrequent_categories_)
        if infrequent is not None
    ],
    index=[c for c, inf in zip(categorical_cols, encoder.infrequent_categories_) if inf is not None],
).sort_values("merged", ascending=False)

print(f"min_frequency=800 merged rare categories in {len(merged)} of {len(categorical_cols)} columns:")
print(merged.to_string())

min_frequency=800 merged rare categories in 1 of 12 columns:
                  categories  merged  columns out
NAME_INCOME_TYPE           5       1            4


### 4d. Assembly — the linear preprocessor

`ColumnTransformer` routes each group to its branch and concatenates the results; the flags pass through unchanged. Unlisted columns are dropped silently, so an assertion checks the four groups cover every column.

`set_output(transform="pandas")` keeps real feature names — `numeric__AMT_CREDIT` rather than a position.

In [43]:
from sklearn.compose import ColumnTransformer

# nothing may fall through: the four groups must account for every column
routed = numeric_cols + ORDINAL_COLS + categorical_cols + flag_cols
assert set(routed) == set(X_train.columns), (
    f"unrouted columns would be dropped: {set(X_train.columns) - set(routed)}"
)

preprocessor_linear = ColumnTransformer([
    ("numeric", numeric_branch, numeric_cols),
    ("ordinal", ordinal_branch, ORDINAL_COLS),
    ("categorical", categorical_branch, categorical_cols),
    ("flags", "passthrough", flag_cols),
]).set_output(transform="pandas")

#preprocessor_linear

### 4e. The tree preprocessor

The same four groups, three of them treated differently.

**Numeric passes through untouched, gaps included.** `DecisionTreeClassifier` and `RandomForestClassifier` can handle missing values natively, and XGBoost and LightGBM are built around it. Keeping the gap is strictly more informative than filling it: the model can still isolate the pensioners in `EMPLOYMENT_YEARS` instead of finding them merged into the 4.5-year bucket.

**No scaling.** A split is a comparison against a threshold, so rescaling moves the threshold and changes nothing else.

**Categoricals become integer codes instead of dummies.** `OrdinalEncoder` gives one column per feature rather than 48. That is not just narrower — it fixes a real distortion in Random Forest: under `max_features` sampling, a one-hot `ORGANIZATION_TYPE` gets 14 chances to be offered at each split while `EXT_SOURCE_2` gets one, which inflates it in exactly the feature-importance ranking the model is there to produce.

The codes carry no meaningful order, which for Decision Tree and Random Forest means a split can only ever separate `code < k` from `code >= k`. XGBoost and LightGBM avoid that by being told the columns are categorical (by casting them to pandas `category` dtype before fitting).

`min_frequency=800` is kept for the same reason as in the linear branch, and no imputer is needed: §2a already left these 13 columns with no gaps.

In [44]:
preprocessor_tree = ColumnTransformer([
    ("numeric", "passthrough", numeric_cols),
    ("ordinal", OrdinalEncoder(
        categories=[EDUCATION_ORDER],
        handle_unknown="use_encoded_value",
        unknown_value=-1,
    ), ORDINAL_COLS),
    ("categorical", OrdinalEncoder(
        min_frequency=800,
        handle_unknown="use_encoded_value",
        unknown_value=-1,
    ), categorical_cols),
    ("flags", "passthrough", flag_cols),
]).set_output(transform="pandas")

#preprocessor_tree

## 5. Saving

### 5a. What each preprocessor produces

Both are fitted on a copy, so the objects saved below stay as definitions. `03_modelling.ipynb` fits them inside each cross-validation fold.

In [45]:
for label, blueprint in [("linear", preprocessor_linear), ("tree", preprocessor_tree)]:
    out = clone(blueprint).fit_transform(X_train)
    print(f"{label:<8}{X_train.shape[1]:>4} -> {out.shape[1]:>4} features   "
          f"{int(X_train.isna().sum().sum()):>9,} gaps -> {int(out.isna().sum().sum()):>9,}")

linear    60 ->   96 features     792,728 gaps ->         0
tree      60 ->   60 features     792,728 gaps ->   792,728


### 5b. Saving the artifacts

**The cleaned split**, so fold membership is fixed permanently — editing a cleaning rule later would otherwise reshuffle the folds and invalidate any result already reported.

**Both preprocessors**, as definitions: the column groups and the branch each group is routed through.

In [46]:
import os
import joblib

PROCESSED_DIR = "../data/processed"
MODELS_DIR = "../models"
os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

for name, X_part, y_part in [
    ("train", X_train, y_train),
    ("test", X_test, y_test),
]:
    joblib.dump(X_part.assign(TARGET=y_part.values), f"{PROCESSED_DIR}/{name}.joblib")

joblib.dump(preprocessor_linear, f"{MODELS_DIR}/preprocessor_linear.joblib")
joblib.dump(preprocessor_tree, f"{MODELS_DIR}/preprocessor_tree.joblib")

for folder in (PROCESSED_DIR, MODELS_DIR):
    for f in sorted(os.listdir(folder)):
        size_kb = os.path.getsize(f"{folder}/{f}") / 1e3
        size, unit = (size_kb / 1e3, "MB") if size_kb >= 1e3 else (size_kb, "KB")
        print(f"{folder + '/' + f:<34}{size:>7.1f} {unit}")

../data/processed/test.joblib        25.7 MB
../data/processed/train.joblib      102.9 MB
../models/preprocessor_linear.joblib    2.8 KB
../models/preprocessor_tree.joblib    2.1 KB


In [47]:
from sklearn.utils.validation import check_is_fitted
from sklearn.exceptions import NotFittedError

for name in ("preprocessor_linear", "preprocessor_tree"):
    reloaded = joblib.load(f"{MODELS_DIR}/{name}.joblib")
    try:
        check_is_fitted(reloaded)
        unfitted = False
    except NotFittedError:
        unfitted = True
    width = clone(reloaded).fit_transform(X_train).shape[1]
    print(f"{name:<22} reloads unfitted: {unfitted}   fits to {width} columns")

preprocessor_linear    reloads unfitted: True   fits to 96 columns
preprocessor_tree      reloads unfitted: True   fits to 60 columns
